In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import torch
from tqdm import tqdm
from stark_qa import load_skb

In [13]:
load_dotenv('../db.env', override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [9]:
def chunks(xs, n: int = 10_000):
    n = max(1, n)
    return [xs[i:i + n] for i in range(0, len(xs), n)]

In [10]:
def create_embeddings(data: dict[int, str], save_dir: str, embedding_name: str, chunk_size = 1_000, filter_out_strs: set[str] = set()) -> None:
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    temp_backup_dir = os.path.join(save_dir, f"{embedding_name}_temp_backup")
    if not os.path.exists(temp_backup_dir):
        os.makedirs(temp_backup_dir)
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    ids = list(data.keys())
    with tqdm(total=len(ids)) as pbar:
        for i, chunk in enumerate(chunks(ids, chunk_size)):
            file_path = os.path.join(temp_backup_dir, f'embeddings{i}.pt')
            if not os.path.exists(file_path):
                good_ids = [idx for idx in chunk if type(data[idx]) == str and data[idx] not in filter_out_strs]
                if len(good_ids) > 0:
                    texts = [data[idx] for idx in good_ids]
                    x = client.embeddings.create(
                        model="text-embedding-ada-002",
                        input=texts,
                        encoding_format="float"
                    )
                    embeddings = {good_ids[j] : torch.tensor([x_data.embedding]) for j, x_data in enumerate(x.data)}
                else:
                    embeddings = {}
                torch.save(embeddings, file_path)
            pbar.update(chunk_size)
            
    all_embeddings = {}
    for file_path in os.listdir(temp_backup_dir):
        all_embeddings.update(torch.load(os.path.join(temp_backup_dir, file_path)))
    torch.save(all_embeddings, os.path.join(save_dir, f"{embedding_name}.pt"))

In [17]:
# Encode author names in stark mag.
dataset_name = 'mag'
skb = load_skb(dataset_name, download_processed=True, root=None)

author_name_data = {skb.node_info[idx]['new_id'] : skb.node_info[idx]['DisplayName'] for idx in skb.get_node_ids_by_type('author')}
save_dir = f"{dataset_name}/text-embedding-ada-002/doc/"
create_embeddings(data=author_name_data, save_dir=save_dir, embedding_name='author_name_emb_dict', chunk_size=1_000, filter_out_strs={'-1'})

# CREATE VECTOR INDEX Institution IF NOT EXISTS
# FOR (i:Institution)
# ON i.nameEmbedding
# OPTIONS { indexConfig: {
#  `vector.dimensions`: 1536,
#  `vector.similarity_function`: 'cosine'
# }}

1105000it [01:25, 12893.27it/s]                             


In [ ]:
# Encode institution names and fields of study. stark mag, as well
dataset_name = 'mag'
skb = load_skb(dataset_name, download_processed=True, root=None)

save_dir = f"./{dataset_name}/text-embedding-ada-002/doc/"

institution_name_data = {skb.node_info[idx]['new_id'] : skb.node_info[idx]['DisplayName'] for idx in skb.get_node_ids_by_type('institution')}
field_of_study_name_data = {skb.node_info[idx]['new_id'] : skb.node_info[idx]['DisplayName'] for idx in skb.get_node_ids_by_type('field_of_study')}

create_embeddings(data=institution_name_data, save_dir=save_dir, embedding_name='institution_name_emb_dict', chunk_size=1_000, filter_out_strs={'-1'})
create_embeddings(data=field_of_study_name_data, save_dir=save_dir, embedding_name='field_of_study_name_emb_dict', chunk_size=1_000, filter_out_strs={'-1'})

In [ ]:
# Encode names of all entities in stark prime.
dataset_name = 'prime'
skb = load_skb(dataset_name, download_processed=True, root=None)

name_data = {k : v['name'] for k,v in skb.node_info.items()}
save_dir = f"{dataset_name}/text-embedding-ada-002/doc/"
create_embeddings(data=author_name_data, save_dir=save_dir, embedding_name='entity_name_emb_dict', chunk_size=1_000, filter_out_strs=set())